In [107]:
%pip install requests==2.32.3 beautifulsoup4==4.12.3 pandas==2.2.2 xhtml2pdf==0.2.16

Note: you may need to restart the kernel to use updated packages.


In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Obtaining Rulemaking File Numbers

In [2]:
def get_rules_df(url):
    # Send a GET request to the URL
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    }
    response = requests.get(url, headers=headers)

    # Check if the request was successful
    if response.status_code == 200:
        # Parse the HTML content using BeautifulSoup
        soup = BeautifulSoup(response.content, 'html.parser')

        # Find the table element
        table = soup.find('table')

        # Extract headers
        headers = [header.text.strip() for header in table.find_all('th')]

        # Extract rows
        rows = []
        for row in table.find_all('tr'):
            columns = row.find_all('td')
            if columns:
                rows.append([column.text.strip() for column in columns])

        # Create a DataFrame using the headers and rows
        df = pd.DataFrame(rows, columns=headers)
        return df
    else:
        print("Failed to retrieve the webpage. Status code:", response.status_code)
        return pd.DataFrame()


In [3]:
# URL of the SEC rulemaking activity page
urls = [f'https://www.sec.gov/rules-regulations/rulemaking-activity?page={x}' for x in range(10)]

dfs = []

for x in urls:
    dfs = dfs+[get_rules_df(x)]

df=pd.concat(dfs).reset_index(drop=True)

In [4]:
# Cleaning up Status Column
df[["Status", "Status Note"]] = df["Status"].str.replace("\n                      ", " ").str.split("\n                  \n", expand = True)

In [5]:
# Removing rows with no 
df = df.loc[df["File Number"]!=""]

In [6]:
df.head()

,Issue Date,File Number,Rulemaking,Status,Status Note
0,"July 1, 2024",S7-16-23,Registration for Index-Linked Annuities; Amend...,Final Rule,Registration for Index-Linked Annuities and Re...
2,"June 21, 2024",S7-05-23,Regulation S-P: Privacy of Consumer Financial ...,Final Rule,Final rule; correction\n\n\n34-100155A \n\n\n\...
3,"May 16, 2024",S7-05-23,Regulation S-P: Privacy of Consumer Financial ...,Final Rule,Regulation S-P: Privacy of Consumer Financial ...
4,"May 13, 2024",S7-2024-02,Customer Identification Programs for Registere...,Proposed Rule,Customer Identification Programs for Registere...
5,"March 27, 2024",S7-13-23,Exemption for Certain Investment Advisers Oper...,Final Rule,Exemption for Certain Investment Advisers Oper...


In [7]:
# We only need unique file numbers because there is only one web-page for 
file_nos = df[["File Number"]].drop_duplicates()

In [9]:
file_nos.shape

(221, 1)

In [10]:
file_nos["File Number Url String1"] = file_nos["File Number"].str.lower()
file_nos["File Number Url String2"] = file_nos["File Number Url String1"].str.replace("-","")

In [11]:
file_nos.head()

,File Number,File Number Url String1,File Number Url String2
0,S7-16-23,s7-16-23,s71623
2,S7-05-23,s7-05-23,s70523
4,S7-2024-02,s7-2024-02,s7202402
5,S7-13-23,s7-13-23,s71323
6,S7-21-21,s7-21-21,s72121


# Obtaining Comment Documents Links

In [86]:
comment_urls = (
    "https://www.sec.gov/comments/"
    +file_nos["File Number Url String1"]
    +"/"
    +file_nos["File Number Url String2"]
    ).to_list()

In [87]:
comment_urls[0]

'https://www.sec.gov/comments/s7-16-23/s71623'

In [94]:
def get_comments_table(url):
    # Send a GET request to the URL
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
    }
    full_url = url+".htm"
    response = requests.get(full_url, headers=headers)

    # Check if the request was successful
    if response.status_code != 200:
        full_url = url+".shtml"
        response = requests.get(full_url, headers=headers)

    # Check if the request was successful
    if response.status_code == 200:
        print(full_url)
        # Parse the HTML content using BeautifulSoup
        soup = BeautifulSoup(response.content, 'html.parser')

        # Obtain all the rows and the associated links
        rows = []
        for row in soup.find_all('tr'):
            a = row.find('a', href=True)
            columns = row.find_all('td')
            if columns:
                if a is not None:
                    rows.append([column.text.strip() for column in columns]+[a['href']])
                else:
                    rows.append([column.text.strip() for column in columns]+[None])

        # Create a DataFrame
        df = pd.DataFrame(rows)

        # Dropping rows with no comment document
        df = df.dropna(subset=2, axis=0, how="all")

        # Keep rows before the "SEC Staff Studies and Reports" row
        check = df[df.iloc[:,0].str.startswith("SEC Staff Studies and Reports")].index
        if len(check) > 0:
            index_studies = check[0]
            df = df.loc[(df.index<index_studies)]

        
        # Keep rows before the "Meetings with SEC Officials" row
        check = df[df.iloc[:,0].str.startswith("Meetings with SEC Officials")].index
        if len(check) > 0:
            index_meetings = check[0]
            df = df.loc[(df.index<index_meetings)]

        # Keeping only rows with comments and the first three columns
        df = df.loc[df.iloc[:,2].str.contains("comments")].iloc[:,0:3]

        # Renaming the columns
        df.columns = ["Date", "Name", "Sublink"]
        
        return df

    else:
        print(full_url)
        print("Failed to retrieve the webpage. Status code:", response.status_code)
        return pd.DataFrame()
        
            

In [95]:
# URL of the SEC Rulemaking Comments
comm_dfs = []
for x in comment_urls:
    comm_dfs = comm_dfs+[get_comments_table(x)]

comm_df=pd.concat(comm_dfs).reset_index(drop=True)

# Creating the Link columns
comm_df["Link"] = "https://www.sec.gov"+comm_df["Sublink"]

https://www.sec.gov/comments/s7-16-23/s71623.htm
https://www.sec.gov/comments/s7-05-23/s70523.htm
https://www.sec.gov/comments/s7-2024-02/s7202402.htm
https://www.sec.gov/comments/s7-13-23/s71323.htm
https://www.sec.gov/comments/s7-21-21/s72121.htm
https://www.sec.gov/comments/s7-29-22/s72922.htm
https://www.sec.gov/comments/s7-10-22/s71022.htm
https://www.sec.gov/comments/s7-02-23/s70223.htm
https://www.sec.gov/comments/s7-2024-01/s7202401.htm
https://www.sec.gov/comments/s7-22-22/s72222.htm
https://www.sec.gov/comments/s7-12-22/s71222.htm
https://www.sec.gov/comments/s7-13-22/s71322.htm
https://www.sec.gov/comments/s7-23-22/s72322.htm
https://www.sec.gov/comments/s7-01-23/s70123.htm
https://www.sec.gov/comments/s7-21-22/s72122.htm
https://www.sec.gov/comments/s7-14-22/s71422.htm
https://www.sec.gov/comments/s7-18-23/s71823.htm
https://www.sec.gov/comments/s7-08-22/s70822.htm
https://www.sec.gov/comments/s7-18-21/s71821.htm
https://www.sec.gov/comments/s7-06-22/s70622.htm
https://www.

In [102]:
comm_df.head(2)

,Date,Name,Sublink,Link
0,"May 2, 2024","Michele Abate, Head of Securities Products & F...",/comments/s7-16-23/s71623-1235274.htm,https://www.sec.gov/comments/s7-16-23/s71623-1...
1,"Apr. 5, 2024","Ronald Coenen Jr., Partner, Eversheds Sutherland",/comments/s7-16-23/s71623-1167914.htm,https://www.sec.gov/comments/s7-16-23/s71623-1...


In [97]:
# Verifying that there are only 3 file types for Comments: htm, pdf, html
comm_df["Sublink"].str.split(".", expand=True)[1].unique()

array(['htm', 'pdf', 'html', 'txt', 'pdff'], dtype=object)

In [104]:
# There is no PDFF file format. So, we will use PDF instead
comm_df["Sublink"] = comm_df["Sublink"].str.replace(".pdff", ".pdf")
comm_df["Link"] = comm_df["Link"].str.replace(".pdff", ".pdf")

In [108]:
# Verifying that there are only 3 file types for Comments: htm, pdf, html
comm_df["Sublink"].str.split(".", expand=True)[1].value_counts()

1
htm     34383
pdf     22278
html       42
txt         8
Name: count, dtype: int64

# Downloading HTML, HTM Pages as PDFs

# Downloading Only PDF Comment Documents

In [225]:
import os

In [229]:
if not os.path.exists("pdfs"):
    os.mkdir("pdfs")

def download_pdf(url):
    headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
        }
    response = requests.get(url, headers=headers)

    with open(os.path.join("pdfs",url.split("/")[-1]), 'wb') as file:
        # Write the content of the response (the downloaded file) to the local file
        file.write(response.content)
        print(f'{url.split("/")[-1]}: Done')



In [230]:
[download_pdf(x) for x in comm_df["Link"] if "pdf" in x]

s71822-445979-1141042.pdf: Done
s71623-321139-834462.pdf: Done
s71623-303499-781402.pdf: Done
s71623-303559-781542.pdf: Done
s71623-303339-781222.pdf: Done
s71623-303439-781302.pdf: Done
s71623-303459-781322.pdf: Done
s71623-303279-781182.pdf: Done
s71623-297139-724642.pdf: Done
s71623-291099-709682.pdf: Done
s71623-292039-711342.pdf: Done
s71623-266579-640482.pdf: Done
s71623-267439-638642.pdf: Done
s71822-445979-1141042.pdf: Done
s71623-321139-834462.pdf: Done
s71623-303499-781402.pdf: Done
s71623-303559-781542.pdf: Done
s71623-303339-781222.pdf: Done
s71623-303439-781302.pdf: Done
s71623-303459-781322.pdf: Done
s71623-303279-781182.pdf: Done
s71623-297139-724642.pdf: Done
s71623-291099-709682.pdf: Done
s71623-292039-711342.pdf: Done
s71623-266579-640482.pdf: Done
s71623-267439-638642.pdf: Done
s71822-445979-1141042.pdf: Done
s71623-321139-834462.pdf: Done
s71623-303499-781402.pdf: Done
s71623-303559-781542.pdf: Done
s71623-303339-781222.pdf: Done
s71623-303439-781302.pdf: Done
s7162

ChunkedEncodingError: ('Connection broken: IncompleteRead(184683 bytes read, 93560 more expected)', IncompleteRead(184683 bytes read, 93560 more expected))